# Clase 01 — Pensar estadísticamente

**Curso:** Estadística Aplicada con Python · **Bloque 1:** Describir los datos
**Tiempo estimado:** 90 minutos · **Capítulo del manual:** `manual/01_pensar_estadisticamente.md`

## Qué vas a ver en esta clase

- La diferencia entre **población, muestra, parámetro y estadístico**, con datos que conocemos por completo.
- Por qué dos muestras de la misma población dan resultados distintos (**variabilidad de muestreo**) y cómo depende del tamaño de la muestra.
- Cómo distinguir **señal** de **ruido**.
- Por qué un **sesgo de muestreo** no desaparece aunque tengas muchísimos datos.

## Requisitos

- Curso *Python para la Ciencia de Datos* (NumPy y Pandas básicos).
- Librerías: `numpy`, `pandas`, `matplotlib` (vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento, escribe (o piensa) qué esperas que ocurra; después compara con lo que ves.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla** del generador aleatorio. Con la misma semilla, cada estudiante obtiene exactamente los mismos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. Una población que conocemos por completo

En la vida real no conoces la población. Aquí la **generamos nosotros**, así que conocemos la verdad y podremos comprobar cuánto se equivoca cada muestra.

Simulamos los sueldos mensuales (en €) de 100 000 empleados. Es una distribución asimétrica, como suelen ser los sueldos reales. *(Datos sintéticos: ejemplo ilustrativo.)*

In [ ]:
N = 100_000
poblacion = rng.lognormal(mean=7.7, sigma=0.4, size=N)    # sueldos mensuales sintéticos (€)

mu = poblacion.mean()        # PARÁMETRO: la media de TODA la población
sigma = poblacion.std()      # PARÁMETRO: su desviación estándar
print(f"Parámetros de la población:  μ = {mu:,.0f} €    σ = {sigma:,.0f} €")

plt.hist(poblacion, bins=80, color="steelblue", edgecolor="white")
plt.axvline(mu, color="crimson", linestyle="--", label=f"μ = {mu:,.0f} €")
plt.title("Población completa: 100 000 sueldos")
plt.xlabel("Sueldo mensual (€)"); plt.ylabel("Empleados"); plt.legend()
plt.show()

---
## 2. Tomar una muestra: el estadístico frente al parámetro

Ahora hacemos lo que haríamos en la realidad: preguntar solo a 200 empleados al azar y calcular su media. Ese número es un **estadístico**. Después comparamos tres muestras distintas de la misma población.

In [ ]:
n = 200
filas = []
for nombre in ["A", "B", "C"]:
    muestra = rng.choice(poblacion, size=n, replace=False)     # muestra aleatoria sin reemplazo
    filas.append({"muestra": nombre, "x̄ (estadístico)": muestra.mean(), "error x̄ − μ": muestra.mean() - mu})

tabla = pd.DataFrame(filas).set_index("muestra").round(1)
print(f"Parámetro real: μ = {mu:,.1f} €")
tabla

**Observa:** las tres medias son distintas entre sí y ninguna es exactamente μ. Ninguna está "mal": es lo esperable.

**Pregunta para ti:** si repitieras esto 1 000 veces, ¿cómo crees que se repartirían las medias? ¿Alrededor de qué valor? ¿Cuánto de ancho sería el abanico? Anótalo antes de seguir.

---
## 3. Experimento guiado: la distribución de las medias muestrales

Repetimos el muestreo 2 000 veces (siempre con n = 200) y guardamos cada media. Así vemos no un resultado, sino **todos los resultados posibles**.

In [ ]:
REPS = 2_000
n = 200

# cada fila es una muestra de tamaño n; la media por filas nos da 2 000 estadísticos
muestras = rng.choice(poblacion, size=(REPS, n))          # con reemplazo: aproximación válida (n << N)
medias = muestras.mean(axis=1)

print(f"Media de las {REPS} medias : {medias.mean():,.1f} €   (μ = {mu:,.1f} €)")
print(f"Desviación de las medias  : {medias.std():,.1f} €   ← cuánto oscila un estadístico")
print(f"Rango observado           : de {medias.min():,.0f} € a {medias.max():,.0f} €")

plt.hist(medias, bins=50, color="darkorange", edgecolor="white")
plt.axvline(mu, color="crimson", linestyle="--", label="μ real")
plt.title(f"Distribución de {REPS} medias muestrales (n = {n})")
plt.xlabel("Media de la muestra (€)"); plt.ylabel("Frecuencia"); plt.legend()
plt.show()

**¿Qué observas?** Anota tus conclusiones:

- ¿Las medias se reparten alrededor de μ, o alrededor de otro valor?
- La forma del histograma de medias, ¿se parece a la de la población (asimétrica) o es distinta?
- Si en la vida real tuvieras *una sola* muestra, ¿cuánto te podrías equivocar como mucho, más o menos?

*(Tu respuesta aquí.)*

### Efecto del tamaño de la muestra

Repetimos el experimento con muestras de distinto tamaño y medimos cuánto oscilan las medias. Comparamos con el valor **σ/√n**, que veremos con detalle en la clase 6.

In [ ]:
filas = []
for n in [10, 50, 200, 1_000, 5_000]:
    medias_n = rng.choice(poblacion, size=(REPS, n)).mean(axis=1)
    filas.append({"n": n,
                  "oscilación observada": medias_n.std(),
                  "σ/√n (teórico)": sigma / np.sqrt(n)})

efecto_n = pd.DataFrame(filas).set_index("n").round(1)
display(efecto_n)

plt.loglog(efecto_n.index, efecto_n["oscilación observada"], "o-", label="observada")
plt.loglog(efecto_n.index, efecto_n["σ/√n (teórico)"], "--", label="σ/√n")
plt.title("Cuánto oscila la media muestral según el tamaño de la muestra")
plt.xlabel("Tamaño de la muestra n"); plt.ylabel("Desviación de las medias (€)"); plt.legend()
plt.show()

**¿Qué observas?**

- Al pasar de n = 50 a n = 200 (cuatro veces más datos), ¿la oscilación se reduce a la mitad? ¿Y de n = 200 a n = 5 000?
- ¿Compensa siempre recoger más datos? ¿Qué coste tiene reducir el ruido a la mitad?

*(Tu respuesta aquí.)*

---
## 4. Tipos de variables: no todo número es numérico

`pandas` decide el tipo según cómo vienen los datos, no según su significado. Veamos qué pasa con un código postal.

In [ ]:
df = pd.DataFrame({
    "cp": [28001, 15702, 8001, 41001],                    # código postal (¡el 08001 perdió su cero!)
    "hijos": [0, 2, 1, 3],                                # numérica discreta
    "satisfaccion": ["alta", "baja", "media", "alta"],    # categórica ordinal
    "sueldo": [2100.5, 1850.0, 2500.75, 3100.2],          # numérica continua
})
print(df.dtypes, end="\n\n")
print("Media de los códigos postales (¡no tiene ningún sentido!):", df["cp"].mean())

# Le decimos a pandas lo que cada columna es de verdad
df["cp"] = df["cp"].astype(str).str.zfill(5).astype("category")
df["satisfaccion"] = pd.Categorical(df["satisfaccion"], categories=["baja", "media", "alta"], ordered=True)
print(df.dtypes)
df

**Observa:** antes de resumir cualquier columna hay que decidir de qué tipo es. Los códigos postales son *categorías* que casualmente se escriben con dígitos, y la satisfacción es *ordinal*: tiene un orden (`baja < media < alta`), pero no podemos decir que la distancia entre "baja" y "media" sea la misma que entre "media" y "alta".

---
## 5. Señal y ruido

Una empresa prueba dos versiones de una página web. Los tiempos de compra (minutos) tienen una media de 3,2 y una desviación de 0,8. Usaremos muestras de 50 usuarios por versión.

**Escenario 1: no hay ninguna diferencia real.** Las dos versiones tienen la misma distribución. Aun así, ¿cuánto pueden diferir las medias de dos muestras?

In [ ]:
REPS = 5_000
n = 50
a = rng.normal(3.2, 0.8, size=(REPS, n)).mean(axis=1)     # versión A
b = rng.normal(3.2, 0.8, size=(REPS, n)).mean(axis=1)     # versión B: MISMA distribución
dif_sin_efecto = b - a

print(f"Diferencia típica entre medias cuando NO hay diferencia real: ±{dif_sin_efecto.std():.2f} min")
print(f"Proporción de veces que |B − A| ≥ 0,20 min por puro azar : {(np.abs(dif_sin_efecto) >= 0.20).mean():.1%}")

plt.hist(dif_sin_efecto, bins=60, color="slategray", edgecolor="white")
plt.axvline(0, color="black")
plt.title("Diferencia de medias B − A cuando no hay ninguna diferencia real")
plt.xlabel("B − A (minutos)"); plt.ylabel("Frecuencia")
plt.show()

**Observa:** aunque no haya ningún efecto, las diferencias de medias oscilan alrededor de 0 y, con cierta frecuencia, llegan a 0,2 minutos o más. Ver una diferencia de 0,2 min en *una* muestra no prueba nada por sí solo.

**Escenario 2: B es realmente 0,2 minutos más rápida.**

In [ ]:
b_real = rng.normal(3.0, 0.8, size=(REPS, n)).mean(axis=1)      # B con una mejora real de 0,2 min
dif_con_efecto = b_real - a

print(f"Diferencia media observada (debería estar cerca de −0,20): {dif_con_efecto.mean():+.3f} min")
print(f"Proporción de veces que B parece MÁS LENTA que A (B − A > 0), siendo realmente más rápida: {(dif_con_efecto > 0).mean():.1%}")

plt.hist(dif_sin_efecto, bins=60, alpha=0.6, label="sin efecto real", color="slategray")
plt.hist(dif_con_efecto, bins=60, alpha=0.6, label="B 0,2 min más rápida", color="seagreen")
plt.axvline(0, color="black")
plt.title("Señal y ruido: las dos distribuciones se solapan")
plt.xlabel("B − A (minutos)"); plt.legend()
plt.show()

**¿Qué observas?** Las dos distribuciones **se solapan**. Con muestras de 50 usuarios, una mejora real de 0,2 min se confunde a menudo con el ruido: incluso puede pasar que B *parezca* peor. Distinguir señal de ruido es exactamente el objetivo de los intervalos de confianza y los tests de hipótesis (clases 6 a 9).

---
## 6. Sesgo de muestreo: más datos no lo arreglan

Una tienda quiere conocer la satisfacción media (de 0 a 10) de sus 100 000 clientes. Pero solo responden a la encuesta los clientes más satisfechos: la probabilidad de responder **crece con la satisfacción**. Es un caso de **autoselección**. *(Datos sintéticos: ejemplo ilustrativo.)*

In [ ]:
N = 100_000
satisfaccion = np.clip(rng.normal(6.0, 2.0, size=N), 0, 10)
p_responder = 1 / (1 + np.exp(-(satisfaccion - 7)))                   # los más contentos responden más
responde = rng.random(N) < p_responder
respondientes = satisfaccion[responde]

mu_sat = satisfaccion.mean()
print(f"Satisfacción media REAL de la población      : {mu_sat:.2f}")
print(f"Clientes que responden a la encuesta         : {responde.sum():,} de {N:,}")
print(f"Satisfacción media entre los que responden   : {respondientes.mean():.2f}")

Ahora comparamos dos formas de obtener una muestra de tamaño *n*: **(a)** al azar entre todos los clientes y **(b)** entre quienes responden (sesgada). Medimos el error absoluto medio de la media muestral repitiendo el proceso 300 veces.

In [ ]:
filas = []
for n in [50, 500, 5_000, 20_000]:
    err_azar, err_sesgo = [], []
    for _ in range(300):
        err_azar.append(abs(rng.choice(satisfaccion, n, replace=False).mean() - mu_sat))
        err_sesgo.append(abs(rng.choice(respondientes, n, replace=False).mean() - mu_sat))
    filas.append({"n": n, "error muestra aleatoria": np.mean(err_azar), "error muestra sesgada": np.mean(err_sesgo)})

sesgo = pd.DataFrame(filas).set_index("n").round(3)
display(sesgo)

sesgo.plot(marker="o", logx=True, figsize=(8, 4), title="Error de la media según el tamaño de la muestra")
plt.ylabel("Error absoluto medio"); plt.show()

**¿Qué observas?**

- El error de la muestra **aleatoria** cae al crecer *n* (es solo ruido).
- El error de la muestra **sesgada** se queda estancado en torno al mismo valor: ese es el sesgo. Con 20 000 respuestas te sigues equivocando lo mismo que con 500.

**Conclusión:** el tamaño de la muestra reduce el ruido, pero no el sesgo. Antes de preguntar *"¿cuántos datos tengo?"*, pregunta *"¿cómo llegaron a mi tabla y a quién representan?"*.

---
## Resumen de la clase

1. **Población / muestra / parámetro / estadístico:** el estadístico estima el parámetro, y casi nunca coincide con él.
2. **Variabilidad de muestreo:** cada muestra da un valor distinto; la oscilación baja como 1/√n (cuatro veces más datos, la mitad de ruido).
3. **Tipo de variable:** decide qué resumen tiene sentido; un número no siempre es una variable numérica.
4. **Señal y ruido:** una diferencia observada en una muestra puede ser puro azar; las distribuciones de "hay efecto" y "no hay efecto" se solapan.
5. **Sesgo de muestreo:** más datos no lo corrigen; hay que revisar cómo se obtuvo la muestra.

**Siguiente paso:** Clase 2 — Estadística descriptiva, donde aprenderás a resumir una muestra con media, mediana, dispersión y cuantiles. Y ahora, a practicar: `notebooks_practica/clase01_practica_pensar_estadisticamente.ipynb`.